# Module 1: Terraform - Provisioning the Platform Substrate

## Overview

This notebook walks through deploying AWS infrastructure using the **official `langchain-ai/terraform` repository**.

### Key Principles

- ✅ Use the **official** Terraform repo (do not fork)
- ✅ Pin module versions for reproducibility
- ✅ Use remote state & locking
- ✅ Plan before applying
- ✅ Capture outputs needed for Helm

### What We'll Deploy

- Amazon EKS cluster
- RDS PostgreSQL
- ElastiCache Redis
- S3 bucket for blob storage
- IAM roles and policies
- EBS CSI driver addon

**Estimated time:** 45-60 minutes


In [3]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path so we can import shared as a package
# Find the notebooks directory by looking for the shared folder
possible_paths = [
    Path.cwd().parent,  # If cwd is module-1, go up one level to notebooks
    Path.cwd(),  # If cwd is already notebooks
    Path.cwd() / "notebooks",  # If cwd is workspace root
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

# Add notebooks directory to path so 'shared' can be imported as a package
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])
print(f"\nArtifacts directory: {artifacts_dir}")


Bootstrapping workshop environment...
✅ Loaded environment variables from workshop.env
### Checking required tools...
✅ aws is available
✅ terraform is available
✅ helm is available
✅ kubectl is available
✅ jq is available
✅ All required tools are available
### AWS Configuration
Region: us-east-1
Account ID: 640174622193
User ARN: arn:aws:sts::640174622193:assumed-role/AWSReservedSSO_AdministratorAccess_8d292fb20bd59b88/cory
User ID: AROAZKDLBQXY3D7XOHYJN:cory
✅ AWS credentials are valid
✅ Artifacts directory ready: /Users/cory/repos/langchain-ai/langsmith-self-hosted-workshops/notebooks/module-1/artifacts
✅ Bootstrap complete!

Artifacts directory: /Users/cory/repos/langchain-ai/langsmith-self-hosted-workshops/notebooks/module-1/artifacts


## Understanding the Official Terraform Repository

The `langchain-ai/terraform` repository contains modules for deploying LangSmith infrastructure. We use the **official** repository because:

1. **Support:** Support will expect to see standard configurations
2. **Updates:** Official modules receive security and feature updates
3. **Documentation:** Official modules are documented and tested
4. **Compatibility:** Ensures compatibility with Helm charts

**Important:** We do **not** fork the upstream repository. We reference it directly.


In [4]:
import os
import re
from pathlib import Path
from shared._validation import require_env, ok, warn, fail
from shared._shell import run

def expand_env_vars(path_str: str) -> str:
    """Expand environment variable references in a path string."""
    # Expand $VAR and ${VAR} references
    def replace_var(match):
        var_name = match.group(1) or match.group(2)
        return os.environ.get(var_name, match.group(0))
    
    # Replace $VAR and ${VAR} patterns
    path_str = re.sub(r'\$\{([^}]+)\}|\$([a-zA-Z_][a-zA-Z0-9_]*)', replace_var, path_str)
    return path_str

# Get required configuration
config = require_env("TERRAFORM_DIR", "CLUSTER_NAME", "AWS_REGION", "WORKSHOP_NAME")

# Expand environment variables in the path (e.g., $TERRAFORM_REPO_DIR, $HOME)
terraform_dir_str = expand_env_vars(config["TERRAFORM_DIR"])
terraform_dir = Path(terraform_dir_str).expanduser().resolve()

cluster_name = config["CLUSTER_NAME"]
region = config["AWS_REGION"]
workshop_name = config["WORKSHOP_NAME"]

print("### Terraform Configuration")
print(f"Terraform Directory: {terraform_dir}")
print(f"Cluster Name: {cluster_name}")
print(f"Region: {region}")
print(f"Workshop Name: {workshop_name}\n")

if not terraform_dir.exists():
    fail(f"Terraform directory does not exist: {terraform_dir}")
    print("\n💡 To fix this:")
    print("   1. Clone the official Terraform repository:")
    print("      git clone https://github.com/langchain-ai/terraform.git <target-directory>")
    print("   2. Update TERRAFORM_DIR in your .env file to point to:")
    print(f"      TERRAFORM_DIR=\"<target-directory>/aws/langsmith\"")
    print("   3. Run this notebook again")
    raise RuntimeError(f"Terraform directory not found: {terraform_dir}")

ok(f"Terraform directory exists: {terraform_dir}")

# Check Terraform version
print("\n### Terraform Version")
result = run(["terraform", "version"], check=True, stream=False)
print(result.stdout)


### Terraform Configuration
Terraform Directory: /Users/cory/repos/langchain-ai/terraform/modules/aws/langsmith
Cluster Name: langsmith-workshop
Region: us-east-1
Workshop Name: langsmith-self-hosted-operator

✅ Terraform directory exists: /Users/cory/repos/langchain-ai/terraform/modules/aws/langsmith

### Terraform Version
Terraform v1.14.2
on darwin_arm64

Your version of Terraform is out of date! The latest version
is 1.14.3. You can update by downloading from https://developer.hashicorp.com/terraform/install



## Identifying the Correct Module Path

The Terraform repository is organized by cloud provider and deployment type. For AWS self-hosted deployments, we need the AWS module.

**Typical path structure:**
```
terraform/
  modules/
    aws/
      langsmith/    # <-- This is the module we use
        main.tf
        variables.tf
        outputs.tf
        ...
```

Let's verify the module structure.


In [5]:
# Verify Terraform module structure
print("### Terraform Module Structure\n")

# Check for key files
key_files = ["main.tf", "variables.tf", "outputs.tf"]
found_files = []

for file in key_files:
    file_path = terraform_dir / file
    if file_path.exists():
        found_files.append(file)
        ok(f"Found {file}")
    else:
        warn(f"Missing {file}")

if len(found_files) == len(key_files):
    ok("Terraform module structure looks correct")
else:
    warn("Some expected Terraform files are missing")
    print("💡 Ensure TERRAFORM_DIR points to the correct module path (e.g., terraform/aws/langsmith)")

# List all .tf files for reference
print("\n### All Terraform Files in Module")
tf_files = sorted(terraform_dir.glob("*.tf"))
for tf_file in tf_files:
    print(f"  - {tf_file.name}")


### Terraform Module Structure

✅ Found main.tf
✅ Found variables.tf
✅ Found outputs.tf
✅ Terraform module structure looks correct

### All Terraform Files in Module
  - backend.tf
  - main.tf
  - outputs.tf
  - variables.tf
  - versions.tf


## Pinning Module Versions

**Critical:** Always pin Terraform module versions for reproducibility. This ensures:
- Consistent deployments across environments
- Predictable behavior
- Ability to roll back if needed

Check the `versions.tf` or `main.tf` file to see what versions are pinned.


In [6]:
# Check for version constraints
print("### Checking Module Version Constraints\n")

versions_file = terraform_dir / "versions.tf"
if versions_file.exists():
    print("Found versions.tf:")
    print("=" * 60)
    with open(versions_file) as f:
        print(f.read())
    print("=" * 60)
else:
    # Check main.tf for version constraints
    main_file = terraform_dir / "main.tf"
    if main_file.exists():
        with open(main_file) as f:
            content = f.read()
            if "required_version" in content or "version" in content.lower():
                print("Version constraints found in main.tf:")
                print("=" * 60)
                # Show relevant lines
                for i, line in enumerate(content.split('\n'), 1):
                    if 'version' in line.lower() or 'required' in line.lower():
                        print(f"{i:4}: {line}")
                print("=" * 60)
            else:
                warn("No version constraints found")
                print("💡 Consider adding version constraints to ensure reproducibility")


### Checking Module Version Constraints

Found versions.tf:
terraform {
  required_version = ">= 1.0"

  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.100"
    }
    helm = {
      source  = "hashicorp/helm"
      version = "~> 2.16"
    }
    kubernetes = {
      source  = "hashicorp/kubernetes"
      version = "~> 2.37"
    }
  }
}



## Remote State & Locking

**Why remote state matters:**
- Enables team collaboration
- Prevents concurrent modifications
- Provides state backup and recovery

**Why locking matters:**
- Prevents state corruption from concurrent runs
- Required for production deployments

Check if remote state backend is configured.


In [7]:
# Check for backend configuration
print("### Checking Backend Configuration\n")

backend_file = terraform_dir / "backend.tf"
if backend_file.exists():
    print("Found backend.tf:")
    print("=" * 60)
    with open(backend_file) as f:
        print(f.read())
    print("=" * 60)
    ok("Backend configuration found")
else:
    # Check for backend block in other files
    backend_configs = []
    for tf_file in terraform_dir.glob("*.tf"):
        with open(tf_file) as f:
            content = f.read()
            if "backend" in content:
                backend_configs.append(tf_file.name)
    
    if backend_configs:
        print(f"Backend configuration found in: {', '.join(backend_configs)}")
        ok("Backend configuration exists")
    else:
        warn("No backend configuration found")
        print("💡 For production, configure remote state (S3 + DynamoDB for locking)")
        print("   For workshops, local state may be acceptable")


### Checking Backend Configuration

Found backend.tf:
# You may want to keep the terraform state in S3 instead of locally
# terraform {
#   backend "s3" {
#     bucket         = "langsmith-terraform-state-bucket"
#     key            = "envs/dev/terraform.tfstate"  # customize as needed
#     region         = "us-west-2"
#     encrypt        = true
#   }
# }

✅ Backend configuration found


## Terraform Initialization

Before planning or applying, Terraform must be initialized. This downloads providers and modules.


In [8]:
# Initialize Terraform
print("### Initializing Terraform\n")
print("This may take a few minutes as it downloads providers and modules...\n")

result = run(
    ["terraform", "init"],
    cwd=str(terraform_dir),
    check=True,
    stream=True
)

ok("Terraform initialization complete")


### Initializing Terraform

This may take a few minutes as it downloads providers and modules...

Initializing the backend...
Initializing modules...
- eks in ../eks
- eks.eks in .terraform/modules/eks.eks
- eks.eks.eks_managed_node_group in .terraform/modules/eks.eks/modules/eks-managed-node-group
- eks.eks.eks_managed_node_group.user_data in .terraform/modules/eks.eks/modules/_user_data
- eks.eks.fargate_profile in .terraform/modules/eks.eks/modules/fargate-profile
- eks.eks.kms in .terraform/modules/eks.eks.kms
- eks.eks.self_managed_node_group in .terraform/modules/eks.eks/modules/self-managed-node-group
- eks.eks.self_managed_node_group.user_data in .terraform/modules/eks.eks/modules/_user_data
- eks.eks_blueprints_addons in .terraform/modules/eks.eks_blueprints_addons
- eks.eks_blueprints_addons.argo_events in .terraform/modules/eks.eks_blueprints_addons.argo_events
- eks.eks_blueprints_addons.argo_rollouts in .terraform/modules/eks.eks_blueprints_addons.argo_rollouts
- eks.eks_b

## Planning vs Applying

**Always plan before applying.** The plan shows:
- What resources will be created/modified/destroyed
- Any configuration errors
- Estimated costs (if configured)

**Review the plan carefully** before proceeding to apply.


In [ ]:
# Create terraform plan
plan_file = artifacts_dir / "terraform-plan.txt"

print("### Creating Terraform Plan\n")
print("This will show what resources Terraform intends to create/modify/destroy.\n")
print("⚠️  Review the plan carefully before applying!\n")

# Collect Terraform variables from environment
terraform_vars = []
postgres_username = os.environ.get("POSTGRES_USERNAME", "").strip()
postgres_password = os.environ.get("POSTGRES_PASSWORD", "").strip()

print("### Terraform Variables\n")
missing_vars = []

if postgres_username:
    terraform_vars.extend(["-var", f"postgres_username={postgres_username}"])
    print(f"✅ POSTGRES_USERNAME: {postgres_username}")
else:
    missing_vars.append("POSTGRES_USERNAME")
    warn("POSTGRES_USERNAME not set in environment")

if postgres_password:
    terraform_vars.extend(["-var", f"postgres_password={postgres_password}"])
    print(f"✅ POSTGRES_PASSWORD: {'*' * len(postgres_password)} (hidden)")
else:
    missing_vars.append("POSTGRES_PASSWORD")
    warn("POSTGRES_PASSWORD not set in environment")

if missing_vars:
    print(f"\n❌ Missing required environment variables: {', '.join(missing_vars)}")
    print("💡 To fix this:")
    print("   1. Add these variables to your .env file (or workshop.env):")
    for var in missing_vars:
        print(f"      {var}=\"your-value-here\"")
    print("   2. Re-run the bootstrap cell (first cell) to reload environment variables")
    print("   3. Re-run this cell")
    raise RuntimeError(f"Missing required Terraform variables: {', '.join(missing_vars)}")

print(f"\n✅ All required variables are set. Passing {len(terraform_vars) // 2} variable(s) to Terraform.\n")

# Build terraform plan command
plan_cmd = ["terraform", "plan", "-out=tfplan"] + terraform_vars

result = run(
    plan_cmd,
    cwd=str(terraform_dir),
    check=False,  # Don't fail if plan has warnings
    stream=True
)

# Save plan output
with open(plan_file, "w") as f:
    f.write(result.stdout)
    if result.stderr:
        f.write("\n\nSTDERR:\n")
        f.write(result.stderr)

print(f"\n💡 Plan output saved to: {plan_file}")

if result.returncode == 0:
    ok("Terraform plan completed successfully")
    print("\n⚠️  Review the plan above. If it looks correct, proceed to the next cell to apply.")
else:
    warn(f"Terraform plan had issues (rc={result.returncode})")
    print("💡 Review the errors above before proceeding")


### Creating Terraform Plan

This will show what resources Terraform intends to create/modify/destroy.

⚠️  Review the plan carefully before applying!

var.postgres_password
  Password for the postgres database

  Enter a value: var.postgres_username
  Username for the postgres database

  Enter a value: ╷
│ Error: No value for required variable
│ 
│   on variables.tf line 104:
│  104: variable "postgres_username" {
│ 
│ The root module input variable "postgres_username" is not set, and has no
│ default value. Use a -var or -var-file command line argument to provide a
│ value for this variable.
╵
╷
│ Error: No value for required variable
│ 
│   on variables.tf line 109:
│  109: variable "postgres_password" {
│ 
│ The root module input variable "postgres_password" is not set, and has no
│ default value. Use a -var or -var-file command line argument to provide a
│ value for this variable.
╵

💡 Plan output saved to: /Users/cory/repos/langchain-ai/langsmith-self-hosted-workshops/notebooks/

## Applying Terraform

**⚠️ WARNING:** This will create real AWS resources and incur costs.

Only proceed if:
1. ✅ You've reviewed the plan
2. ✅ You're using the correct AWS account/region
3. ✅ You understand the costs involved

**Estimated deployment time:** 15-30 minutes (EKS cluster creation takes time)


In [ ]:
# Apply Terraform
# ⚠️  UNCOMMENT THE CODE BELOW TO ACTUALLY APPLY
# This is commented out by default to prevent accidental deployments

print("### Applying Terraform Configuration\n")
print("⚠️  This cell is currently DISABLED to prevent accidental deployments.\n")
print("To apply Terraform, uncomment the code below and run this cell.\n")

# UNCOMMENT TO APPLY:
# print("Applying Terraform... This will take 15-30 minutes.\n")
# result = run(
#     ["terraform", "apply", "tfplan"],
#     cwd=str(terraform_dir),
#     check=True,
#     stream=True
# )
# 
# ok("Terraform apply completed successfully")
# 
# # Save apply output
# apply_file = artifacts_dir / "terraform-apply.txt"
# with open(apply_file, "w") as f:
#     f.write(result.stdout)
#     if result.stderr:
#         f.write("\n\nSTDERR:\n")
#         f.write(result.stderr)
# 
# print(f"\n💡 Apply output saved to: {apply_file}")

print("💡 To apply, edit this cell and uncomment the code above")


## Interpreting Terraform Outputs

After Terraform applies successfully, we need to capture the outputs. These outputs contain information needed for Helm deployment:

- Cluster name and endpoint
- RDS connection details
- Redis connection details
- S3 bucket name
- IAM role ARNs

Let's retrieve and save these outputs.


In [ ]:
import json

# Get Terraform outputs
print("### Terraform Outputs\n")

result = run(
    ["terraform", "output", "-json"],
    cwd=str(terraform_dir),
    check=True,
    stream=False
)

outputs = json.loads(result.stdout)

# Save outputs to artifacts
outputs_file = artifacts_dir / "terraform-outputs.json"
with open(outputs_file, "w") as f:
    json.dump(outputs, f, indent=2)

print("Terraform outputs:")
print("=" * 60)
for key, value in outputs.items():
    if isinstance(value, dict) and "value" in value:
        # Terraform outputs are wrapped in {"value": ...}
        val = value["value"]
        if isinstance(val, str) and len(val) > 100:
            print(f"{key}: {val[:100]}... (truncated)")
        else:
            print(f"{key}: {val}")
    else:
        print(f"{key}: {value}")
print("=" * 60)

ok(f"Outputs saved to: {outputs_file}")
print("\n💡 These outputs will be needed for Helm deployment in the next notebook")


## Verifying Infrastructure

Let's verify that the key infrastructure components were created successfully.


In [ ]:
from shared._aws_helpers import eks_cluster_exists, aws_region

region = aws_region()

# Verify EKS cluster
print("### Verifying EKS Cluster\n")
if eks_cluster_exists(cluster_name):
    ok(f"Cluster '{cluster_name}' exists")
    
    # Get cluster endpoint
    result = run(
        ["aws", "eks", "describe-cluster", "--name", cluster_name, "--region", region, "--output", "json"],
        check=True,
        stream=False
    )
    cluster_info = json.loads(result.stdout)["cluster"]
    print(f"  Status: {cluster_info['status']}")
    print(f"  Endpoint: {cluster_info['endpoint']}")
    print(f"  Version: {cluster_info['version']}")
else:
    warn(f"Cluster '{cluster_name}' not found")

# Verify kubectl access
print("\n### Configuring kubectl\n")
try:
    run(
        ["aws", "eks", "update-kubeconfig", "--name", cluster_name, "--region", region],
        check=True,
        stream=True
    )
    ok("kubectl configured")
    
    # Test cluster access
    result = run(
        ["kubectl", "cluster-info"],
        check=True,
        stream=False
    )
    print(result.stdout)
except Exception as e:
    warn(f"Could not configure kubectl: {e}")

# Check for RDS (if output available)
if "rds" in str(outputs).lower() or "postgres" in str(outputs).lower():
    print("\n### RDS PostgreSQL\n")
    print("💡 Verify RDS instance is available in AWS console")
    print("   Check outputs above for connection details")

# Check for ElastiCache (if output available)
if "redis" in str(outputs).lower() or "elasticache" in str(outputs).lower():
    print("\n### ElastiCache Redis\n")
    print("💡 Verify Redis cluster is available in AWS console")
    print("   Check outputs above for connection details")

# Check for S3 bucket
if "s3" in str(outputs).lower() or "bucket" in str(outputs).lower():
    print("\n### S3 Bucket\n")
    print("💡 Verify S3 bucket exists in AWS console")
    print("   Check outputs above for bucket name")


## Summary

### ✅ What We Accomplished

- [ ] Initialized Terraform
- [ ] Created and reviewed Terraform plan
- [ ] Applied Terraform configuration (if you uncommented the apply step)
- [ ] Captured Terraform outputs
- [ ] Verified infrastructure components

### 📋 Key Takeaways

1. **Use official Terraform repo** - Don't fork, reference directly
2. **Pin versions** - Ensures reproducibility
3. **Use remote state** - Required for production
4. **Always plan first** - Review before applying
5. **Save outputs** - Needed for Helm deployment

### 🎯 Next Steps

Proceed to **03_helm_install_langsmith.ipynb** to install LangSmith using Helm.

**Important:** Make sure you have:
- ✅ Terraform outputs saved
- ✅ Cluster accessible via kubectl
- ✅ LangSmith license key ready
